# Exercise 2.3 — Eigenphase Repulsion in the $2 \times 2$ CUE

**Chapter 2: Random Matrix Theory** &nbsp;|&nbsp; *Section 2.4: The Circular Unitary Ensemble*

---

## Motivation

The eigenvalues of a Haar-random unitary matrix are not independent — they **repel** each other.  This level repulsion is one of the most striking predictions of random matrix theory and is observed throughout quantum chaos, from nuclear spectra to the Riemann zeta function.  The $2 \times 2$ CUE is the simplest setting where this phenomenon can be derived exactly: the repulsion is **quadratic** (proportional to the spacing squared), which is the hallmark of unitary symmetry.

## Exercise Statement

A $2 \times 2$ CUE matrix (Haar-random $U \in U(2)$) has eigenvalues $e^{i\theta_1}, e^{i\theta_2}$ with joint density proportional to the Vandermonde squared:

$$
P(\theta_1, \theta_2) \propto |e^{i\theta_1} - e^{i\theta_2}|^2.
$$

**(a)** Show $|e^{i\theta_1} - e^{i\theta_2}|^2 = 2 - 2\cos(\theta_1 - \theta_2)$.

**(b)** With $\varphi = \theta_1 - \theta_2$, show the spacing density is $P(\varphi) = (1 - \cos\varphi)/(2\pi)$.

**(c)** Verify $P(0) = 0$ (level repulsion) and find the most probable spacing.

## Solution

### Part (a): Algebraic identity

Expand the modulus squared:

$$
|e^{i\theta_1} - e^{i\theta_2}|^2 = (e^{i\theta_1} - e^{i\theta_2})(e^{-i\theta_1} - e^{-i\theta_2})
$$

$$
= 1 - e^{i(\theta_1 - \theta_2)} - e^{-i(\theta_1 - \theta_2)} + 1 = 2 - 2\cos(\theta_1 - \theta_2). \quad \checkmark
$$

Key feature: the expression depends only on the **spacing** $\varphi = \theta_1 - \theta_2$, not on the individual phases.

### Part (b): Marginal spacing density

Change variables to the spacing $\varphi = \theta_1 - \theta_2$ and the center of mass $\Theta = (\theta_1 + \theta_2)/2$.

The joint density is $P(\varphi, \Theta) \propto 2 - 2\cos\varphi$.  Integrating over the center of mass $\Theta \in [0, 2\pi)$ gives a factor $2\pi$.

The normalization constant is

$$
Z = \int_0^{2\pi} (2 - 2\cos\varphi)\, d\varphi = [2\varphi - 2\sin\varphi]_0^{2\pi} = 4\pi.
$$

Therefore the normalized spacing density is

$$
\boxed{P(\varphi) = \frac{1 - \cos\varphi}{2\pi}.}
$$

### Part (c): Level repulsion and most probable spacing

**Level repulsion:** At $\varphi = 0$:

$$
P(0) = \frac{1 - 1}{2\pi} = 0.
$$

Eigenphases **cannot coincide** — there is a strict vanishing of the density at zero spacing.  Near $\varphi = 0$, $P(\varphi) \approx \varphi^2/(4\pi)$: the repulsion is **quadratic** ($\beta = 2$ for the CUE).

**Most probable spacing:** Maximize $P(\varphi) = (1 - \cos\varphi)/(2\pi)$:

$$
\frac{dP}{d\varphi} = \frac{\sin\varphi}{2\pi} = 0 \quad \Rightarrow \quad \varphi = \pi.
$$

Eigenphases prefer to sit **diametrically opposite** on the unit circle — the maximally separated configuration.  This is the most basic manifestation of spectral rigidity.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

t1, t2, phi = sp.symbols('theta_1 theta_2 phi', real=True)

# Part (a)
expr = sp.expand(sp.Abs(sp.exp(sp.I*t1) - sp.exp(sp.I*t2))**2)
expr_simplified = sp.trigsimp(sp.expand_complex(expr))
print(f"|e^{{iθ₁}} − e^{{iθ₂}}|² = {expr_simplified}")

# Part (b): normalization
Z = sp.integrate(2 - 2*sp.cos(phi), (phi, 0, 2*sp.pi))
print(f"Z = ∫(2−2cosφ)dφ = {Z}")
P = (1 - sp.cos(phi)) / (2*sp.pi)
norm = sp.integrate(P, (phi, 0, 2*sp.pi))
print(f"∫ P(φ) dφ = {norm}")
assert norm == 1

---
## Numerical Verification

In [ ]:
import numpy as np
from scipy.stats import unitary_group

np.random.seed(42)

# Sample eigenphase spacings from Haar-random U(2)
n_samples = 50000
spacings = []
for _ in range(n_samples):
    U = unitary_group.rvs(2)
    phases = np.sort(np.angle(np.linalg.eigvals(U)))
    phi = (phases[1] - phases[0]) % (2*np.pi)
    spacings.append(phi)

spacings = np.array(spacings)

# Histogram vs analytical P(φ)
bins = np.linspace(0, 2*np.pi, 50)
centers = (bins[:-1] + bins[1:]) / 2
hist, _ = np.histogram(spacings, bins=bins, density=True)
P_theory = (1 - np.cos(centers)) / (2*np.pi)

residual = np.max(np.abs(hist - P_theory))
print(f"Max |histogram − P(φ)| = {residual:.4f}")
assert residual < 0.02

# Check P(0) → 0: fraction of spacings < 0.1
frac_small = np.mean(spacings < 0.1)
print(f"Fraction with φ < 0.1: {frac_small:.4f} (level repulsion)")

# Most probable spacing
mode = centers[np.argmax(hist)]
print(f"Most probable spacing: {mode:.2f} (expected π = {np.pi:.2f})")
print("\nLevel repulsion confirmed. ✓")